# Phase 1 — Day 3: Data Cleaning and I/O

**Date:** 2026-04-22

Welcome to Day 3! Today we tackle the messy side of data science: cleaning dirty data and reading/writing files in different formats.

**Learning objectives:**
- Handle missing values (detect, drop, fill)
- Find and remove duplicate rows
- Convert column data types
- Parse and dump JSON
- Read and write CSV, JSON, and Excel files

In [2]:
# Setup
import pandas as pd
import numpy as np
import json
import os
import tempfile

print("All imports ready!")

All imports ready!


In [3]:
# Sample dirty data — we'll clean this throughout the notebook
dirty_data = {
    "name": ["Alice", "Bob", "Charlie", "Alice", "Eve", "Frank", None, "Grace"],
    "age": [25, None, 35, 25, 28, "forty", 22, 30],
    "salary": [50000.0, 60000.0, None, 50000.0, 72000.0, 55000.0, 48000.0, None],
    "city": ["Istanbul", "Ankara", "Istanbul", "Istanbul", "Izmir", "Ankara", "Izmir", "Istanbul"],
    "join_date": ["2023-01-15", "2023-02-20", "2023-03-10", "2023-01-15", "2023-04-05", "bad-date", "2023-06-01", "2023-07-12"]
}

df = pd.DataFrame(dirty_data)
print("Our messy DataFrame:")
df

Our messy DataFrame:


,name,age,salary,city,join_date
0,Alice,25,50000.0,Istanbul,2023-01-15
1,Bob,None,60000.0,Ankara,2023-02-20
2,Charlie,35,NaN,Istanbul,2023-03-10
3,Alice,25,50000.0,Istanbul,2023-01-15
4,Eve,28,72000.0,Izmir,2023-04-05
5,Frank,forty,55000.0,Ankara,bad-date
6,NaN,22,48000.0,Izmir,2023-06-01
7,Grace,30,NaN,Istanbul,2023-07-12


## 1. Missing Values

Real datasets almost always have missing entries. Pandas represents them as `NaN` (Not a Number) or `None`. You need to detect them before you can decide what to do about them.

The three main strategies are: **detect** with `isnull()`, **drop** with `dropna()`, and **fill** with `fillna()`. Which one you pick depends on how much data you can afford to lose and whether a fill value makes sense.

In [4]:
# Detect missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

Missing values per column:
name         1
age          1
salary       2
city         0
join_date    0
dtype: int64

Total missing values: 4


In [6]:
# Drop rows that have ANY missing value
df_dropped = df.dropna()
print(f"Original rows: {len(df)} -> After dropna(): {len(df_dropped)}")
print(df_dropped)
print()

# Drop rows only if ALL values are missing (less aggressive)
df_dropped_all = df.dropna(how="all")
print(f"After dropna(how='all'): {len(df_dropped_all)} rows (no all-NaN rows here)")

Original rows: 8 -> After dropna(): 4
    name    age   salary      city   join_date
0  Alice     25  50000.0  Istanbul  2023-01-15
3  Alice     25  50000.0  Istanbul  2023-01-15
4    Eve     28  72000.0     Izmir  2023-04-05
5  Frank  forty  55000.0    Ankara    bad-date

After dropna(how='all'): 8 rows (no all-NaN rows here)


In [ ]:
# Fill missing values with different strategies
df_filled = df.copy()

# Fill name with "Unknown"
df_filled["name"] = df_filled["name"].fillna("Unknown")

# Fill salary with the median (robust to outliers)
median_salary = df_filled["salary"].median()
df_filled["salary"] = df_filled["salary"].fillna(median_salary)
print(f"Filled missing salaries with median: {median_salary}")

# Forward fill: carry the last valid value forward
print("\nForward fill example on salary column:")
print(df["salary"].ffill())

print("\nDataFrame after targeted fills:")
df_filled

## 2. Duplicates

Duplicate rows sneak into datasets from merges, re-runs, or buggy ETL pipelines. Pandas gives you `duplicated()` to find them and `drop_duplicates()` to remove them.

By default these methods check all columns. You can also pass `subset=["col1", "col2"]` to check only specific columns. The `keep` parameter controls which copy survives: `"first"` (default), `"last"`, or `False` (drop all copies).

In [ ]:
# Find duplicate rows
print("Duplicate rows (True = duplicate):")
print(df.duplicated())
print(f"\nTotal duplicates: {df.duplicated().sum()}")

# Show which rows are duplicates
print("\nThe duplicate rows:")
print(df[df.duplicated(keep=False)])  # keep=False marks ALL copies

In [ ]:
# Remove duplicates
df_no_dupes = df.drop_duplicates()
print(f"Before: {len(df)} rows -> After drop_duplicates(): {len(df_no_dupes)} rows")

# Remove duplicates based on subset of columns
df_no_dupes_name = df.drop_duplicates(subset=["name"], keep="first")
print(f"\nUnique by name only: {len(df_no_dupes_name)} rows")
print(df_no_dupes_name[["name", "age"]])

## 3. Data Type Conversion

Pandas often guesses column types when loading data, and sometimes it guesses wrong. A column of numbers might come in as strings if even one value is non-numeric. You need to convert types explicitly using `astype()`, `pd.to_numeric()`, and `pd.to_datetime()`.

The `errors` parameter is your best friend here. Set it to `"coerce"` to turn unparseable values into `NaN` instead of crashing your script.

In [ ]:
# Check current dtypes
print("Current dtypes:")
print(df.dtypes)
print()

# The 'age' column is 'object' because "forty" is in there
# pd.to_numeric with errors='coerce' turns bad values into NaN
df_clean = df.copy()
df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")
print(f"Age dtype after to_numeric: {df_clean['age'].dtype}")
print(f"'forty' became: {df_clean.loc[5, 'age']}")  # NaN

In [ ]:
# Convert dates with errors='coerce'
df_clean["join_date"] = pd.to_datetime(df_clean["join_date"], errors="coerce")
print(f"join_date dtype: {df_clean['join_date'].dtype}")
print(f"'bad-date' became: {df_clean.loc[5, 'join_date']}")  # NaT (Not a Time)
print()

# Now you can do date math!
df_clean["days_since_join"] = (pd.Timestamp("2026-04-22") - df_clean["join_date"]).dt.days
print("Days since joining:")
print(df_clean[["name", "join_date", "days_since_join"]])

In [ ]:
# astype() for simple conversions
prices = pd.Series(["10", "20", "30", "40"])
print(f"Before: {prices.dtype}")  # object (string)

prices_int = prices.astype(int)
print(f"After astype(int): {prices_int.dtype}")
print(prices_int * 2)  # Now math works!

# Convert int to category (useful for memory savings on low-cardinality columns)
df_clean["city_cat"] = df_clean["city"].astype("category")
print(f"\ncity memory as object: {df_clean['city'].memory_usage()} bytes")
print(f"city memory as category: {df_clean['city_cat'].memory_usage()} bytes")

## 4. JSON Parse and Dump

JSON is the most common format for API responses and config files. Python's built-in `json` module handles it. The key functions are `json.loads()` (string to dict), `json.dumps()` (dict to string), `json.load()` (file to dict), and `json.dump()` (dict to file).

Pandas also has `pd.read_json()` and `df.to_json()` for going directly between JSON and DataFrames.

In [ ]:
# json.loads() — parse a JSON string into a Python dict
json_string = '{"name": "Deniz", "role": "data scientist", "skills": ["python", "sql", "ml"]}'
data = json.loads(json_string)
print(type(data))  # <class 'dict'>
print(data["skills"])  # ['python', 'sql', 'ml']

# json.dumps() — convert a Python dict back to a JSON string
output = json.dumps(data, indent=2)
print(output)

In [ ]:
# json.dump() and json.load() — write to / read from files
tmp_dir = tempfile.mkdtemp()

# Write JSON to a file
with open(os.path.join(tmp_dir, "profile.json"), "w") as f:
    json.dump(data, f, indent=2)

# Read JSON from a file
with open(os.path.join(tmp_dir, "profile.json"), "r") as f:
    loaded = json.load(f)

print("Loaded from file:", loaded)
print(f"Same data? {loaded == data}")

In [ ]:
# Pandas JSON: list of records -> DataFrame
records = [
    {"product": "Laptop", "price": 1200, "stock": 50},
    {"product": "Mouse", "price": 25, "stock": 200},
    {"product": "Keyboard", "price": 75, "stock": 150}
]

# Write and read via pandas
json_path = os.path.join(tmp_dir, "products.json")
pd.DataFrame(records).to_json(json_path, orient="records", indent=2)

df_products = pd.read_json(json_path)
print(df_products)
print(f"\nOrient options: 'records', 'columns', 'index', 'split', 'values'")

## 5. File I/O: CSV, JSON, and Excel

Pandas makes reading and writing tabular files a one-liner. The most common formats are CSV (comma-separated), JSON, and Excel (.xlsx). Each has its own `read_*` and `to_*` functions.

Key parameters to know: `index=False` when writing (to avoid saving the row index as a column), `encoding="utf-8"` for non-ASCII characters, and `sheet_name` for Excel files with multiple tabs.

In [ ]:
# Create sample data for I/O demos
sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=5),
    "product": ["Widget", "Gadget", "Widget", "Doohickey", "Gadget"],
    "revenue": [100, 250, 150, 80, 300],
    "units": [10, 5, 15, 8, 6]
})
print(sales)
print(f"\n--- CSV ---")

In [ ]:
# CSV: Write and Read
csv_path = os.path.join(tmp_dir, "sales.csv")
sales.to_csv(csv_path, index=False)

# Read it back
df_csv = pd.read_csv(csv_path)
print("Read from CSV:")
print(df_csv)
print(f"date dtype: {df_csv['date'].dtype}")  # object! Dates need parse_dates

# Read with date parsing
df_csv2 = pd.read_csv(csv_path, parse_dates=["date"])
print(f"\nWith parse_dates: {df_csv2['date'].dtype}")  # datetime64

In [ ]:
# Excel: Write and Read (requires openpyxl)
try:
    import openpyxl
    excel_path = os.path.join(tmp_dir, "sales.xlsx")
    
    # Write multiple sheets to one Excel file
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        sales.to_excel(writer, sheet_name="All Sales", index=False)
        sales[sales["product"] == "Widget"].to_excel(writer, sheet_name="Widgets Only", index=False)
    
    # Read a specific sheet
    df_widgets = pd.read_excel(excel_path, sheet_name="Widgets Only")
    print("Widgets Only sheet:")
    print(df_widgets)
    
    # Read all sheets at once (returns a dict of DataFrames)
    all_sheets = pd.read_excel(excel_path, sheet_name=None)
    print(f"\nSheet names: {list(all_sheets.keys())}")
except ImportError:
    print("openpyxl not installed. Run: pip install openpyxl")

## Tricky Bits

These are common mistakes that trip up beginners. Run each cell and read the error messages carefully.

In [ ]:
# Trick 1: fillna() doesn't modify in place by default!
s = pd.Series([1, None, 3])
s.fillna(0)  # This does NOTHING to s
print(f"After s.fillna(0) without assignment: {s.tolist()}")  # Still has None!

# Fix: assign back or use inplace=True
s = s.fillna(0)
print(f"After s = s.fillna(0): {s.tolist()}")  # Now it's fixed

In [ ]:
# Trick 2: NaN comparisons are always False
import numpy as np

val = np.nan
print(f"np.nan == np.nan: {val == val}")  # False! NaN is not equal to itself
print(f"Use pd.isna() instead: {pd.isna(val)}")  # True

# This means you can't filter NaN with ==
s = pd.Series([1, np.nan, 3])
print(f"\ns[s == np.nan]: {s[s == np.nan].tolist()}")  # Empty!
print(f"s[s.isna()]: {s[s.isna()].tolist()}")  # Correct

In [ ]:
# Trick 3: astype() crashes on bad values, to_numeric() can handle them
mixed = pd.Series(["10", "20", "oops", "40"])

try:
    mixed.astype(int)  # This CRASHES
except ValueError as e:
    print(f"astype(int) error: {e}")

# Safe way:
result = pd.to_numeric(mixed, errors="coerce")
print(f"\nto_numeric with coerce: {result.tolist()}")  # 'oops' becomes NaN

In [ ]:
# Trick 4: json.loads() vs json.load() — one takes a string, the other takes a file
import json

# loads = load STRING
data = json.loads('{"a": 1}')
print(f"loads() takes a string: {data}")

# load = load FILE
try:
    data = json.load('{"a": 1}')  # Wrong! This expects a file object
except AttributeError as e:
    print(f"load() with a string: {e}")
    print("Use loads() for strings, load() for file objects!")

## Trick Questions

Test your understanding. Click to reveal the answers.

**Q1:** What does `df.dropna(how="all")` do differently from `df.dropna()`?

<details>
<summary>Answer</summary>
<code>dropna()</code> (default <code>how="any"</code>) drops a row if ANY value is NaN. <code>dropna(how="all")</code> only drops a row if ALL values in that row are NaN. The "all" version is much less aggressive.
</details>

**Q2:** You run `df["price"] = df["price"].fillna(0)`. Later you discover that 0 is a valid price in your data. Why is this a problem?

<details>
<summary>Answer</summary>
You can no longer tell the difference between "the price was actually 0" and "the price was missing and we filled it with 0." You've lost information. A better approach might be to fill with the median, or keep the NaN and handle it downstream.
</details>

**Q3:** What is the difference between `json.dumps()` and `json.dump()`?

<details>
<summary>Answer</summary>
<code>json.dumps()</code> returns a JSON-formatted string. <code>json.dump()</code> writes JSON directly to a file object. The "s" stands for "string."
</details>

**Q4:** You read a CSV with `pd.read_csv("data.csv")` and the date column shows dtype `object`. Why?

<details>
<summary>Answer</summary>
By default, <code>read_csv</code> does not parse dates. You need to pass <code>parse_dates=["date_column"]</code> to convert it to datetime. Otherwise it stays as a plain string.
</details>

**Q5:** `df.duplicated()` returns `[False, False, False, True]` for 4 rows. Does this mean only 1 row is duplicated?

<details>
<summary>Answer</summary>
No, it means there are 2 identical rows, but only the second one is marked as True (the duplicate). The first occurrence is marked False by default (<code>keep="first"</code>). Use <code>keep=False</code> to mark all copies as True.
</details>

## Exercises

Fill in the `___` blanks and run each cell. The `assert` statements will tell you if you got it right.

In [ ]:
# Exercise 1: Count missing values in the 'salary' column
ex1 = pd.Series([50000, None, 72000, None, 55000])
missing_count = ex1.___().sum()

assert missing_count == 2, f"Expected 2, got {missing_count}"
print(f"Exercise 1 passed! Missing count: {missing_count}")

In [ ]:
# Exercise 2: Fill missing ages with the mean age
ex2 = pd.DataFrame({"name": ["A", "B", "C", "D"], "age": [20, None, 30, None]})
mean_age = ex2["age"].mean()
ex2["age"] = ex2["age"].___(mean_age)

assert ex2["age"].isna().sum() == 0, "Still has NaN values!"
assert ex2["age"].tolist() == [20.0, 25.0, 30.0, 25.0], f"Wrong values: {ex2['age'].tolist()}"
print(f"Exercise 2 passed! Filled with mean: {mean_age}")

In [ ]:
# Exercise 3: Remove duplicate rows from this DataFrame
ex3 = pd.DataFrame({"x": [1, 2, 1, 3, 2], "y": [10, 20, 10, 30, 20]})
ex3_clean = ex3.___()

assert len(ex3_clean) == 3, f"Expected 3 rows, got {len(ex3_clean)}"
print(f"Exercise 3 passed! {len(ex3)} rows -> {len(ex3_clean)} unique rows")

In [ ]:
# Exercise 4: Safely convert these strings to numbers (bad values become NaN)
ex4 = pd.Series(["100", "200", "N/A", "400", "error"])
ex4_numeric = pd.to_numeric(ex4, errors=___)

assert ex4_numeric.isna().sum() == 2, f"Expected 2 NaN, got {ex4_numeric.isna().sum()}"
assert ex4_numeric.iloc[0] == 100.0
print(f"Exercise 4 passed! Result: {ex4_numeric.tolist()}")

In [ ]:
# Exercise 5: Parse a JSON string into a Python dictionary
json_str = '{"city": "Istanbul", "population": 15000000}'
result = json.___(json_str)

assert isinstance(result, dict), f"Expected dict, got {type(result)}"
assert result["city"] == "Istanbul"
print(f"Exercise 5 passed! City: {result['city']}, Pop: {result['population']}")

In [ ]:
# Exercise 6: Write a DataFrame to CSV without the index column
ex6 = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
ex6_path = os.path.join(tmp_dir, "ex6.csv")
ex6.to_csv(ex6_path, ___=False)

# Verify: read it back and check there's no 'Unnamed: 0' column
ex6_check = pd.read_csv(ex6_path)
assert "Unnamed: 0" not in ex6_check.columns, "Index column leaked into CSV!"
print(f"Exercise 6 passed! Columns: {ex6_check.columns.tolist()}")

In [ ]:
# Exercise 7: Convert date strings to datetime, coercing bad values to NaT
dates = pd.Series(["2023-01-15", "not-a-date", "2023-03-20"])
dates_clean = pd.to_datetime(dates, errors=___)

assert dates_clean.isna().sum() == 1, f"Expected 1 NaT, got {dates_clean.isna().sum()}"
assert str(dates_clean.iloc[0].date()) == "2023-01-15"
print(f"Exercise 7 passed! {dates_clean.tolist()}")

## Solutions

<details>
<summary>Exercise 1</summary>
<code>missing_count = ex1.isna().sum()</code> — or <code>isnull()</code>, they're the same thing.
</details>

<details>
<summary>Exercise 2</summary>
<code>ex2["age"] = ex2["age"].fillna(mean_age)</code>
</details>

<details>
<summary>Exercise 3</summary>
<code>ex3_clean = ex3.drop_duplicates()</code>
</details>

<details>
<summary>Exercise 4</summary>
<code>pd.to_numeric(ex4, errors="coerce")</code>
</details>

<details>
<summary>Exercise 5</summary>
<code>result = json.loads(json_str)</code> — remember the "s" stands for "string."
</details>

<details>
<summary>Exercise 6</summary>
<code>ex6.to_csv(ex6_path, index=False)</code>
</details>

<details>
<summary>Exercise 7</summary>
<code>pd.to_datetime(dates, errors="coerce")</code>
</details>

## Cumulative Review

Mixed exercises covering Day 1 (Pandas Essentials) and Day 2 (NumPy Vectorization).

In [ ]:
# Review 1 (Day 1 - Pandas): Use .loc to select rows where city is "Istanbul"
review_df = pd.DataFrame({
    "name": ["Ali", "Bea", "Can", "Dila"],
    "city": ["Istanbul", "Ankara", "Istanbul", "Izmir"],
    "score": [85, 90, 78, 92]
})
istanbul = review_df.loc[review_df[___] == "Istanbul"]

assert len(istanbul) == 2
print(f"Review 1 passed! Istanbul rows: {len(istanbul)}")

In [ ]:
# Review 2 (Day 1 - Pandas): Group by city and get the mean score
mean_scores = review_df.groupby(___)[___].mean()

assert mean_scores["Istanbul"] == 81.5
print(f"Review 2 passed!\n{mean_scores}")

In [ ]:
# Review 3 (Day 2 - NumPy): Create a 2x3 array of zeros
arr = np.___((2, 3))

assert arr.shape == (2, 3)
assert arr.sum() == 0.0
print(f"Review 3 passed! Shape: {arr.shape}")

In [ ]:
# Review 4 (Day 2 - NumPy): Vectorized operation — multiply every element by 3
a = np.array([10, 20, 30, 40])
result = a ___ 3

assert result.tolist() == [30, 60, 90, 120]
print(f"Review 4 passed! {result}")

In [ ]:
# Review 5 (Day 2 - NumPy): Calculate the mean along axis=0 (column-wise)
matrix = np.array([[1, 2, 3], [4, 5, 6]])
col_means = matrix.mean(axis=___)

assert col_means.tolist() == [2.5, 3.5, 4.5]
print(f"Review 5 passed! Column means: {col_means}")

In [ ]:
# Review 6 (Day 1 - Pandas): Select rows 1-3 using iloc
review_slice = review_df.iloc[___:___]

assert len(review_slice) == 3
assert review_slice.iloc[0]["name"] == "Bea"
print(f"Review 6 passed! Selected: {review_slice['name'].tolist()}")

### Cumulative Review Solutions

<details>
<summary>Review 1</summary>
<code>istanbul = review_df.loc[review_df["city"] == "Istanbul"]</code>
</details>

<details>
<summary>Review 2</summary>
<code>mean_scores = review_df.groupby("city")["score"].mean()</code>
</details>

<details>
<summary>Review 3</summary>
<code>arr = np.zeros((2, 3))</code>
</details>

<details>
<summary>Review 4</summary>
<code>result = a * 3</code>
</details>

<details>
<summary>Review 5</summary>
<code>col_means = matrix.mean(axis=0)</code> — axis=0 means "collapse rows," giving you one value per column.
</details>

<details>
<summary>Review 6</summary>
<code>review_slice = review_df.iloc[1:4]</code> — iloc uses exclusive end, so 1:4 gives indices 1, 2, 3.
</details>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║            DATA CLEANING & I/O — CHEAT SHEET                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  MISSING VALUES                                              ║
║  df.isnull().sum()          Count NaN per column             ║
║  df.dropna()                Drop rows with any NaN           ║
║  df.dropna(how="all")       Drop rows where ALL are NaN      ║
║  df.fillna(value)           Replace NaN with value            ║
║  df["col"].ffill()          Forward fill                      ║
║                                                              ║
║  DUPLICATES                                                  ║
║  df.duplicated()            Boolean mask of dupes             ║
║  df.drop_duplicates()       Remove duplicate rows             ║
║  df.drop_duplicates(        Check specific columns            ║
║    subset=["col"])                                            ║
║                                                              ║
║  TYPE CONVERSION                                             ║
║  pd.to_numeric(s, errors="coerce")    Safe numeric convert   ║
║  pd.to_datetime(s, errors="coerce")   Safe date convert      ║
║  s.astype(int)              Simple type cast (crashes on bad) ║
║                                                              ║
║  JSON                                                        ║
║  json.loads(string)         String -> dict                    ║
║  json.dumps(dict)           Dict -> string                    ║
║  json.load(file)            File -> dict                      ║
║  json.dump(dict, file)      Dict -> file                      ║
║                                                              ║
║  FILE I/O                                                    ║
║  pd.read_csv("f.csv")              Read CSV                  ║
║  df.to_csv("f.csv", index=False)   Write CSV (no index)      ║
║  pd.read_json("f.json")            Read JSON                 ║
║  df.to_json("f.json")              Write JSON                ║
║  pd.read_excel("f.xlsx")           Read Excel                ║
║  df.to_excel("f.xlsx", index=False) Write Excel              ║
║  pd.read_csv(parse_dates=["col"])   Auto-parse dates         ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---

**Next up: Day 4 — FakerAndPythonCore**

You'll learn to generate synthetic data with Faker and numpy.random, plus sharpen your Python fundamentals: list/dict comprehensions, functions, and error handling with try/except.